# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratheekrao12/-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Lane 2 is a **ranking/scoring problem**: for a human SEO or content reviewer, the model output should order pseudonymized pages from highest to lowest priority for review. The practical decision is which pages enter a limited review queue and which one a reviewer inspects first.

A classification model can support that ranking by estimating the probability of a clearly defined binary outcome, such as `is_declining_label = 1` when the observed snapshot uses `trend_direction == "down"`. These estimated probabilities can sort pages, while a reviewer still decides whether and how to refresh a selected page.

It is important to distinguish the **business decision** from the **modeling technique**. The business decision is ranking pages for review; classification is one possible modeling technique used to produce scores. The later output is still a ranking, and “ranking/scoring” remains the lane type. A score is not itself proof that refreshing a page will cause improvement.

In [1]:
lane = "Lane 2 — Refresh / Content Opportunity Scoring"
task_type = "ranking/scoring"
print({"lane": lane, "task_type": task_type})
assert lane.startswith("Lane 2")
assert task_type == "ranking/scoring"

{'lane': 'Lane 2 — Refresh / Content Opportunity Scoring', 'task_type': 'ranking/scoring'}


## 2. Target or proxy

The introductory proxy is:

```python
is_declining_label = (trend_direction == "down")
```

Here, `1` means that the page's observed snapshot is categorized as `down` according to the current rule; `0` means it is not. This proxy measures whether the page was declining under the repository's existing 30-day trend category. It is useful for learning because it provides an immediately available, page-level binary outcome for practicing classification and ranking on honest observed snapshot data.

The proxy does **not** establish that a page will decline in the future, because it only describes recent measured behavior. It also does not prove that refreshing a page will improve performance: the label is not a treatment effect, and refresh decisions can change subsequent performance for reasons unrelated to the data snapshot.

A stronger label would be future-looking. Later, use a historical feature window to build predictors through an earlier cutoff date, then measure an outcome in a separate later window—for example, whether impressions fall by at least 20% across the subsequent 30 days. Features must come only from the historical window and must not include `trend_direction`, `trend_pct`, future outcome columns, or other fields derived from the outcome window.

The starter file already contains the outcome source `trend_direction`; in this assignment the proxy is derived locally and is not used as a predictive feature.

In [2]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
candidates = []
for root in [cwd, *cwd.parents]:
    candidates.append(root / "data" / "raw" / "content_refresh_anonymized.csv")

csv_path = next((p.resolve() for p in candidates if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

pages = pd.read_csv(csv_path)
raw_shape = pages.shape
pages["is_declining_label"] = (pages["trend_direction"] == "down").astype(int)
print("CSV resolved:", csv_path)
print("Proxy distribution:")
print(pages["is_declining_label"].value_counts().sort_index().rename(index={0: "not down (0)", 1: "down (1)"}))
assert "is_declining_label" in pages.columns
assert pages["is_declining_label"].isin([0, 1]).all()

CSV resolved: /Users/pratheekrao/FlyRank_internship/-flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Proxy distribution:
is_declining_label
not down (0)    13738
down (1)        16262
Name: count, dtype: int64


## 3. Success metric

**Primary metric: Precision@20.** If a reviewer can inspect 20 pages, Precision@20 is the fraction of those 20 ranked pages that are genuinely labeled positive in the held-out evaluation data. **Secondary metric: Precision@50**, using the same definition for the top 50 pages.

These metrics match limited review capacity: they reward a useful list at the actual review cutoff instead of counting every correct decision across all 30,000 pages. Generic accuracy can look high because it rewards correct classifications everywhere, including thousands of pages outside the review queue; it says little about the quality of the few pages a reviewer receives.

Comparison must be fair and honest. First split or hold out a suitable evaluation dataset, freeze it, and rank both a baseline (such as the fixed rule or another simple declared rule) and the candidate model on the same held-out pages. Use one model-independent ranking convention and calculate Precision@20 and Precision@50 on that identical evaluation set. A result is not better until measured; this notebook makes no performance claim and reports no model result.

In [3]:
def precision_at_k(labels, k):
    """Return relevant positives among the top k labels in a ranked list."""
    labels = list(labels)
    if k <= 0:
        raise ValueError("k must be positive")
    if len(labels) < k:
        raise ValueError("The held-out ranked list has fewer pages than k")
    return sum(labels[:k]) / k

print("Metric definition ready: Precision@20 = relevant positives in top 20 / 20")
print("Metric definition ready: Precision@50 = relevant positives in top 50 / 50")

Metric definition ready: Precision@20 = relevant positives in top 20 / 20
Metric definition ready: Precision@50 = relevant positives in top 50 / 50


## 4. The unit of analysis, as a real dataframe

The starter CSV has one row per pseudonymized content page identified by `content_id`. In the code below, the seven displayed columns are public-safe teaching fields: page identifier, 90-day impressions, days since last update, average position, CTR, trend direction, and the proxy label derived from that direction.

In [4]:
required_columns = [
    "content_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "trend_direction", "is_declining_label",
]

safe_preview = pages[required_columns].head(10).copy()
print("Dataframe shape (raw starter data):", pages.shape)
print("Displayed safe slice shape:", safe_preview.shape)
display(safe_preview)

print("Target distribution (is_declining_label):")
print(pages["is_declining_label"].value_counts().sort_index())
print("Content ID unique:", pages["content_id"].is_unique)
print("Duplicate rows:", int(pages.duplicated().sum()))
print("Missing required-column values:")
print(pages[required_columns].isna().sum())

assert raw_shape == (30000, 44)
assert pages.shape == (30000, 45)
assert pages["content_id"].is_unique
assert pages.duplicated().sum() == 0
assert pages[required_columns].isna().sum().sum() == 0

Dataframe shape (raw starter data): (30000, 45)
Displayed safe slice shape: (10, 7)


,content_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,19140,14,44.0,0.13,down,1
5,content_d4084a4bc775,3970,20,8.5,0.03,down,1
6,content_9a34b442b552,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,1724,22,21.2,0.06,stable,0
8,content_5e6c160719bc,32574,20,46.0,0.09,down,1
9,content_c27558df2b0c,1240,104,4.9,0.16,down,1


Target distribution (is_declining_label):
is_declining_label
0    13738
1    16262
Name: count, dtype: int64
Content ID unique: True


Duplicate rows: 0
Missing required-column values:
content_id                0
impressions_90d           0
days_since_last_update    0
avg_position              0
ctr                       0
trend_direction           0
is_declining_label        0
dtype: int64


## 5. Why ML beats a fixed rule here

A fixed rule can create a clear starting point, but a single threshold combination is often too brittle. The example below treats a page as stale and visible only when `days_since_last_update >= 180` and `impressions_90d >= 500`. That rule ignores interactions and patterns among freshness, position, traffic, engagement, content characteristics, and trend. It can miss a high-opportunity page below one threshold, flag a page that deserves no action in context, and use the same cutoff everywhere even when sectors or content types behave differently.

ML may combine many signals, estimate gradual or nonlinear effects, and learn context-dependent combinations that a hand-written threshold misses. However, ML is justified only if it performs better than the fixed rule on appropriate validation and held-out data. A model that merely predicts the same rule is not evidence of learning a real relationship.

There is also a trade-off between interpretability and complexity. A threshold rule is easy to explain, audit, and revise. A more complex model can capture richer patterns, but it needs stronger validation and human-readable reason codes or local explanations so reviewers can understand and challenge its recommendations.

In [5]:
stale_and_visible = (
    (pages["days_since_last_update"] >= 180)
    & (pages["impressions_90d"] >= 500)
)
print("Fixed-rule pages selected:", int(stale_and_visible.sum()))
print("Fraction selected:", round(float(stale_and_visible.mean()), 4))

checks = {
    "lane_selected": lane.startswith("Lane 2"),
    "ranking_framed": task_type == "ranking/scoring",
    "proxy_defined": "is_declining_label" in pages.columns,
    "precision_metric_defined": callable(precision_at_k),
    "raw_dataframe_loaded": raw_shape == (30000, 44),
    "safe_columns_present": all(c in pages.columns for c in required_columns),
    "content_id_unique": pages["content_id"].is_unique,
    "no_duplicate_rows": pages.duplicated().sum() == 0,
    "required_values_present": pages[required_columns].isna().sum().sum() == 0,
}
print("Assignment self-check:")
for name, passed in checks.items():
    print("- {}: {}".format(name, "PASS" if passed else "FAIL"))
assert all(checks.values())

Fixed-rule pages selected: 17
Fraction selected: 0.0006
Assignment self-check:
- lane_selected: PASS
- ranking_framed: PASS
- proxy_defined: PASS
- precision_metric_defined: PASS
- raw_dataframe_loaded: PASS
- safe_columns_present: PASS
- content_id_unique: PASS
- no_duplicate_rows: PASS
- required_values_present: PASS


## Self-check

The Markdown requirements and the final executable cell verify the assignment requirements honestly. No commit or push is performed in this notebook or this assignment step.

Before submission, the assignment requirements are confirmed as follows:

- [x] Every section is filled with Markdown reasoning and supporting executable code.
- [x] The notebook is intended to run top to bottom with no errors after execution.
- [x] No private client names, URLs, or raw queries are displayed; the preview uses pseudonymized identifiers.
- [x] Claims use observed, measured, directional, or decision-support language and avoid causal claims.
- [x] This notebook is not committed or pushed; repository submission remains for the intern after review.